# Ai2020 기반 DFN 모델 파라미터화 및 동일 데이터 정합도 확인

이 노트북은 네 개의 원본 Excel 파일에서 저율 전압, 실측 양극 OCP, 동적 충·방전 데이터를 읽고 최종 파라미터를 적용한 Doyle–Fuller–Newman(DFN) 모델을 다시 계산한다. 저장된 결과 CSV나 로컬 Python 모듈을 사용하지 않는다.

재현 흐름은 다음과 같다.

1. 입력 파일과 PyBaMM 환경 확인
2. C/50 qOCV와 실측 양극 GITT OCP 구성
3. stoichiometry window, geometry, 고체 확산계수, 반응속도 계수 적용
4. 0.5C → 1C → 2C 충·방전 이력을 연속 DFN 실험으로 계산
5. 전체 공통 시간구간과 elapsed-time 10–90% 구간의 RMSE/MAE 평가
6. 충전 중 음극 전위 확인

> 전압 정합도는 아래의 연속 이력으로 새로 계산한 조건별 및 통합 RMSE/MAE로 확인한다. 파라미터 선정과 정합도 확인에 같은 데이터를 사용했으므로 독립 holdout validation은 아니며, 이전 실행 숫자와의 자동 비교나 성공 판정은 하지 않는다.

## 전체 연구 흐름

**공개 Ai2020 모델 확인**
$\rightarrow$ **셀 및 전극 실험값 정리**
$\rightarrow$ **OCP와 stoichiometry window 구성**
$\rightarrow$ **geometry, 고체 확산, 반응속도 반영**
$\rightarrow$ **PyBaMM DFN 연속 실험 실행**
$\rightarrow$ **0.5C·1C·2C 충·방전 전압 정합도 및 내부 전위 확인**

## 실행 환경과 입력 파일

필요한 파일은 다음 네 개다. Colab에서는 아래 설치 셀을 실행한 뒤 파일 선택 창에서 네 파일을 한 번에 업로드한다.

- `260610 저율데이터 1.xlsx`
- `260610 저율데이터 2.xlsx`
- `260808 Eneterch셀 열화데이터 v1 6-6.xlsx`
- `260902 Enertech셀 데이터 모음.xlsx`

In [ ]:
%pip install -q pybamm==26.8.0.0 openpyxl

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pybamm
from IPython.display import display


REQUIRED_PYBAMM_VERSION = "26.8.0.0"
if pybamm.__version__ != REQUIRED_PYBAMM_VERSION:
    raise RuntimeError(
        f"PyBaMM {REQUIRED_PYBAMM_VERSION} is required; found {pybamm.__version__}. "
        "Restart the Colab runtime after the installation cell if necessary."
    )

warnings.filterwarnings("ignore", message="Workbook contains no default style")
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25})

INPUT_DIRECTORY = Path("/content") if Path("/content").exists() else Path.cwd()
REQUIRED_FILES = [
    "260610 저율데이터 1.xlsx",
    "260610 저율데이터 2.xlsx",
    "260808 Eneterch셀 열화데이터 v1 6-6.xlsx",
    "260902 Enertech셀 데이터 모음.xlsx",
]


def upload_missing_files():
    missing = [name for name in REQUIRED_FILES if not (INPUT_DIRECTORY / name).exists()]
    if not missing:
        return
    try:
        from google.colab import files
    except ImportError as error:
        raise FileNotFoundError("Missing input files: " + ", ".join(missing)) from error

    print("Select the four Excel input files.")
    files.upload()
    missing = [name for name in REQUIRED_FILES if not (INPUT_DIRECTORY / name).exists()]
    if missing:
        raise FileNotFoundError("Missing input files after upload: " + ", ".join(missing))


upload_missing_files()
input_paths = {name: INPUT_DIRECTORY / name for name in REQUIRED_FILES}
print("PyBaMM version:", pybamm.__version__)
print("Input directory:", INPUT_DIRECTORY)
for name in REQUIRED_FILES:
    print("✓", name)

In [ ]:
EXPECTED_SHEETS = {
    "260610 저율데이터 1.xlsx": {"record"},
    "260610 저율데이터 2.xlsx": {"record"},
    "260808 Eneterch셀 열화데이터 v1 6-6.xlsx": {"step", "record"},
    "260902 Enertech셀 데이터 모음.xlsx": {"CATHODE_GITT"},
}

for file_name, required_sheets in EXPECTED_SHEETS.items():
    available_sheets = set(pd.ExcelFile(input_paths[file_name]).sheet_names)
    missing_sheets = required_sheets - available_sheets
    if missing_sheets:
        raise ValueError(f"{file_name}: missing sheets {sorted(missing_sheets)}")
    print(f"✓ {file_name}: {sorted(required_sheets)}")

# Chapter 1. 연구 목적 및 Ai2020 모델 선정

## 1.1 DFN 모델이란?

DFN 모델은 전극 두께 방향의 전해질 수송, 입자 내부 고체확산, 전극반응과 전위를 함께 계산한다. 따라서 단자전압뿐 아니라 음극 전위와 농도 상태를 충전 프로토콜의 제약조건으로 검토할 수 있다.

## 1.2 DFN 모델 기반 충전 프로토콜 최적화 방법

최종적으로는 충전시간을 줄이면서 전압, 음극 전위, 농도와 온도 제한을 만족하는 전류 프로파일을 찾는 것이 목적이다. 이 노트북은 최적화 전에 필요한 기준 DFN 모델이 동일 데이터의 전압 거동을 어느 정도 재현하는지 확인한다.

## 1.3 Ai2020 모델 및 대상 셀 선정 근거

PyBaMM의 Ai2020 parameter set은 대상 셀 계열의 DFN 구현을 제공한다. 본 모델은 Ai2020을 기반으로 유지하면서 실측 양극 OCP, 재산정한 inventory, teardown/SEM geometry, GITT 확산계수와 최종 선정한 반응속도 coefficient를 단계적으로 반영한다.

## 1.4 기존 Ai2020 적용 모델의 한계

공개 parameter set을 그대로 사용한 기존 Ai2020 모델은 방전은 상대적으로 잘 맞지만 충전은 충분히 잘 맞지 않아, 충전 프로토콜 최적화의 기준 모델로 사용하기에 한계가 있다. 대상 셀의 양극 OCP, stoichiometry window, geometry와 반응속도 차이가 남기 때문이다. 특히 전압만 맞추기 위해 여러 파라미터를 동시에 조정하면 파라미터 간 상호보상이 생길 수 있으므로, 실험 근거가 있는 값을 우선 고정한다.

## C/50 qOCV와 동적 실험 데이터 읽기

저율 충·방전 반복곡선으로 기준 용량과 qOCV를 계산한다. 동적 파일에서는 0.5C·1C·2C의 CC 충전·방전 branch와 CC 전류, 초기 저율 방전 전류, 각 충전 CV step의 종료전류를 읽는다.

In [ ]:
SOC_GRID = np.linspace(0.0, 1.0, 1001)
TEST_C_RATES = (0.5, 1.0, 2.0)
DYNAMIC_STEP_INDEX = {
    (0.5, True): 7,
    (0.5, False): 10,
    (1.0, True): 12,
    (1.0, False): 15,
    (2.0, True): 17,
    (2.0, False): 20,
}
INITIAL_DISCHARGE_STEP_INDEX = 5
CV_STEP_INDEX = {0.5: 8, 1.0: 13, 2.0: 18}


def clean_curve(horizontal, vertical):
    horizontal = np.asarray(horizontal, dtype=float).reshape(-1)
    vertical = np.asarray(vertical, dtype=float).reshape(-1)
    finite = np.isfinite(horizontal) & np.isfinite(vertical)
    horizontal, vertical = horizontal[finite], vertical[finite]
    order = np.argsort(horizontal, kind="stable")
    horizontal, vertical = horizontal[order], vertical[order]
    unique_horizontal, inverse = np.unique(horizontal, return_inverse=True)
    if len(unique_horizontal) == len(horizontal):
        return horizontal, vertical
    averaged_vertical = np.bincount(inverse, weights=vertical) / np.bincount(inverse)
    return unique_horizontal, averaged_vertical


def mean_over_common_range(curves, number_of_points=3001):
    lower_bound = max(curve[0].min() for curve in curves)
    upper_bound = min(curve[0].max() for curve in curves)
    grid = np.linspace(lower_bound, upper_bound, number_of_points)
    values = np.vstack([np.interp(grid, *curve) for curve in curves])
    return grid, values.mean(axis=0)


def load_low_rate_full_cell(file_paths):
    charge_curves, discharge_curves = [], []
    charge_capacities, discharge_capacities = [], []
    required_columns = {"Step Type", "Capacity(Ah)", "Voltage(V)", "Current(A)"}

    for file_path in file_paths:
        data = pd.read_excel(file_path, sheet_name="record")
        missing_columns = required_columns - set(data.columns)
        if missing_columns:
            raise ValueError(f"{file_path.name}: missing columns {sorted(missing_columns)}")
        contiguous_step = data["Step Type"].astype(str).ne(
            data["Step Type"].astype(str).shift()
        ).cumsum()
        branches = {"CC Chg": [], "CC DChg": []}

        for _, branch in data.groupby(contiguous_step):
            step_type = str(branch["Step Type"].iloc[0])
            if step_type not in branches:
                continue
            capacity = pd.to_numeric(branch["Capacity(Ah)"], errors="coerce").to_numpy(float)
            voltage = pd.to_numeric(branch["Voltage(V)"], errors="coerce").to_numpy(float)
            current = float(pd.to_numeric(branch["Current(A)"], errors="coerce").median())
            finite = np.isfinite(capacity) & np.isfinite(voltage)
            capacity, voltage = capacity[finite], voltage[finite]
            if len(capacity) < 50 or capacity.max() < 2.2 or abs(current) > 0.06:
                continue
            end_capacity = float(capacity.max())
            branch_soc = (
                capacity / end_capacity
                if step_type == "CC Chg"
                else 1.0 - capacity / end_capacity
            )
            branch_soc, voltage = clean_curve(branch_soc, voltage)
            branches[step_type].append((end_capacity, np.interp(SOC_GRID, branch_soc, voltage)))

        for capacity, voltage in branches["CC Chg"]:
            charge_capacities.append(capacity)
            charge_curves.append(voltage)
        # The first discharge follows a partial initial charge and is excluded.
        for capacity, voltage in branches["CC DChg"][1:]:
            discharge_capacities.append(capacity)
            discharge_curves.append(voltage)

    if not charge_curves or not discharge_curves:
        raise ValueError("No complete C/50 charge/discharge curves were found.")
    mean_charge = np.mean(charge_curves, axis=0)
    mean_discharge = np.mean(discharge_curves, axis=0)
    return {
        "charge": mean_charge,
        "discharge": mean_discharge,
        "qocv": 0.5 * (mean_charge + mean_discharge),
        "capacity_ah": float(0.5 * (np.mean(charge_capacities) + np.mean(discharge_capacities))),
        "number_of_charge_curves": len(charge_curves),
        "number_of_discharge_curves": len(discharge_curves),
    }


def load_dynamic_branches(file_path):
    step = pd.read_excel(file_path, sheet_name="step")
    record = pd.read_excel(file_path, sheet_name="record")
    required_step_columns = {"Cycle Index", "Step Index", "Oneset Date", "End Date", "Step Type"}
    required_record_columns = {"Date", "Step Type", "Voltage(V)", "Current(A)"}
    if required_step_columns - set(step.columns):
        raise ValueError("Dynamic step sheet has unexpected columns.")
    if required_record_columns - set(record.columns):
        raise ValueError("Dynamic record sheet has unexpected columns.")
    record["Date"] = pd.to_datetime(record["Date"])
    branches = {}

    def records_for_step(step_index, expected_step_type):
        selected = step[(step["Cycle Index"] == 1) & (step["Step Index"] == step_index)]
        if len(selected) != 1:
            raise ValueError(f"Expected one dynamic step for index {step_index}; found {len(selected)}")
        selected_row = selected.iloc[0]
        if str(selected_row["Step Type"]) != expected_step_type:
            raise ValueError(
                f"Step index {step_index}: expected {expected_step_type}, "
                f"found {selected_row['Step Type']}"
            )
        selected_records = record[
            (record["Date"] >= pd.to_datetime(selected_row["Oneset Date"]))
            & (record["Date"] <= pd.to_datetime(selected_row["End Date"]))
            & (record["Step Type"].astype(str) == str(selected_row["Step Type"]))
        ].copy()
        if selected_records.empty:
            raise ValueError(f"Dynamic records are empty for step index {step_index}")
        return selected_records

    for condition, step_index in DYNAMIC_STEP_INDEX.items():
        expected_step_type = "CC Chg" if condition[1] else "CC DChg"
        branch = records_for_step(step_index, expected_step_type)

        time_minutes = (branch["Date"] - branch["Date"].iloc[0]).dt.total_seconds().to_numpy(float) / 60.0
        branches[condition] = {
            "time_min": time_minutes,
            "voltage_v": pd.to_numeric(branch["Voltage(V)"], errors="coerce").to_numpy(float),
            "current_a": abs(float(pd.to_numeric(branch["Current(A)"], errors="coerce").median())),
        }

    initial_discharge = records_for_step(INITIAL_DISCHARGE_STEP_INDEX, "CC DChg")
    protocol_currents = {
        "initial_discharge_a": abs(float(pd.to_numeric(initial_discharge["Current(A)"], errors="coerce").median())),
        "cv_cutoff_a": {},
    }
    for c_rate, step_index in CV_STEP_INDEX.items():
        cv_current = pd.to_numeric(
            records_for_step(step_index, "CV Chg")["Current(A)"], errors="coerce"
        ).dropna()
        protocol_currents["cv_cutoff_a"][c_rate] = abs(float(cv_current.iloc[-1]))
    return branches, protocol_currents


low_rate_data = load_low_rate_full_cell([
    input_paths["260610 저율데이터 1.xlsx"],
    input_paths["260610 저율데이터 2.xlsx"],
])
experimental_branches, dynamic_protocol_currents = load_dynamic_branches(
    input_paths["260808 Eneterch셀 열화데이터 v1 6-6.xlsx"]
)
reference_capacity_ah = low_rate_data["capacity_ah"]

print(f"C/50 charge curves: {low_rate_data['number_of_charge_curves']}")
print(f"C/50 discharge curves: {low_rate_data['number_of_discharge_curves']}")
print(f"Reference capacity: {reference_capacity_ah:.6f} Ah")
print(f"Initial discharge current: {dynamic_protocol_currents['initial_discharge_a']:.6f} A")
print("CV cutoff currents:", {rate: round(current, 6) for rate, current in dynamic_protocol_currents["cv_cutoff_a"].items()})
if not np.isclose(reference_capacity_ah, 2.356528, atol=5e-4):
    raise ValueError("The reconstructed reference capacity differs from the final model record.")

## 실측 양극 GITT OCP 구성

통합 workbook의 `CATHODE_GITT` 시트에서 7-7과 7-8 반복시험의 pulse 후 휴지전압을 읽고, lithiation/delithiation 공통 구간에서 평균한다.

In [ ]:
FARADAY_CONSTANT = 96485.33212


def build_measured_cathode_ocp(workbook_path, ai2020_parameters):
    frame = pd.read_excel(workbook_path, sheet_name="CATHODE_GITT", header=8)
    required_columns = {"source_channel", "pulse_direction", "Qcum_Ah", "Vrest_V"}
    missing_columns = required_columns - set(frame.columns)
    if missing_columns:
        raise ValueError(f"CATHODE_GITT: missing columns {sorted(missing_columns)}")

    disk_area_m2 = np.pi * (0.01 / 2.0) ** 2
    disk_capacity_ah = (
        FARADAY_CONSTANT
        * float(ai2020_parameters["Maximum concentration in positive electrode [mol.m-3]"])
        * float(ai2020_parameters["Positive electrode active material volume fraction"])
        * float(ai2020_parameters["Positive electrode thickness [m]"])
        * disk_area_m2
        / 3600.0
    )
    lithiation_curves, delithiation_curves = [], []
    for channel in ("7-7", "7-8"):
        channel_data = frame.loc[frame["source_channel"].astype(str).eq(channel)]
        charge = channel_data.loc[
            channel_data["pulse_direction"].eq("charge_pulse"), ["Qcum_Ah", "Vrest_V"]
        ].to_numpy(float)
        discharge = channel_data.loc[
            channel_data["pulse_direction"].eq("discharge_pulse"), ["Qcum_Ah", "Vrest_V"]
        ].to_numpy(float)
        if len(charge) < 2 or len(discharge) < 2:
            raise ValueError(f"CATHODE_GITT channel {channel} has insufficient pulse data.")
        delithiation_curves.append(clean_curve(1.0 - charge[:, 0] / disk_capacity_ah, charge[:, 1]))
        discharge_span = discharge[:, 0].max() / disk_capacity_ah
        lithiation_curves.append(
            clean_curve((1.0 - discharge_span) + discharge[:, 0] / disk_capacity_ah, discharge[:, 1])
        )

    mean_lithiation = mean_over_common_range(lithiation_curves)
    mean_delithiation = mean_over_common_range(delithiation_curves)
    return mean_over_common_range([mean_lithiation, mean_delithiation])


ai2020_parameters = pybamm.ParameterValues("Ai2020")
measured_cathode_ocp = build_measured_cathode_ocp(
    input_paths["260902 Enertech셀 데이터 모음.xlsx"], ai2020_parameters
)
print(
    "Measured cathode OCP support:",
    f"{measured_cathode_ocp[0].min():.4f}–{measured_cathode_ocp[0].max():.4f} stoichiometry",
)

In [ ]:
def evaluate_parameter_function(parameter_values, parameter_name, grid):
    function = parameter_values[parameter_name]
    return np.array([
        float(np.asarray(function(pybamm.Scalar(float(value))).evaluate()).reshape(-1)[0])
        for value in grid
    ])


stoichiometry_grid = np.linspace(0.001, 0.999, 3001)
ai2020_negative_ocp = (
    stoichiometry_grid,
    evaluate_parameter_function(ai2020_parameters, "Negative electrode OCP [V]", stoichiometry_grid),
)
ai2020_positive_ocp = (
    stoichiometry_grid,
    evaluate_parameter_function(ai2020_parameters, "Positive electrode OCP [V]", stoichiometry_grid),
)

final_window = {
    "x0": 0.0038033652717561297,
    "x100": 0.9541670383294881,
    "y0": 0.9707163439556511,
    "y100": 0.4316462299737839,
}


def full_cell_ocv(window, negative_ocp, positive_ocp, soc=SOC_GRID):
    negative_stoichiometry = window["x0"] + soc * (window["x100"] - window["x0"])
    positive_stoichiometry = window["y0"] - soc * (window["y0"] - window["y100"])
    return (
        np.interp(positive_stoichiometry, *positive_ocp)
        - np.interp(negative_stoichiometry, *negative_ocp)
    )


ai2020_ocv = full_cell_ocv(final_window, ai2020_negative_ocp, ai2020_positive_ocp)
measured_cathode_ocv = full_cell_ocv(final_window, ai2020_negative_ocp, measured_cathode_ocp)
fitting_region = (SOC_GRID >= 0.02) & (SOC_GRID <= 0.98)

qocv_summary = pd.DataFrame([
    {
        "OCP combination": "Ai2020 negative + Ai2020 positive",
        "RMSE 2–98% [mV]": 1000.0 * np.sqrt(np.mean((ai2020_ocv[fitting_region] - low_rate_data["qocv"][fitting_region]) ** 2)),
    },
    {
        "OCP combination": "Ai2020 negative + measured positive",
        "RMSE 2–98% [mV]": 1000.0 * np.sqrt(np.mean((measured_cathode_ocv[fitting_region] - low_rate_data["qocv"][fitting_region]) ** 2)),
    },
])
display(qocv_summary.round(3))

figure, axes = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
axes[0].plot(SOC_GRID * 100, low_rate_data["qocv"], color="black", lw=2, label="Experiment")
axes[0].plot(
    SOC_GRID * 100, ai2020_ocv, linestyle="none", marker="s", markevery=45,
    markersize=3.2, color="#6b7280", label="Ai2020 OCP pair",
)
axes[0].plot(
    SOC_GRID * 100, measured_cathode_ocv, linestyle="none", marker="o", markevery=45,
    markersize=3.2, color="#d97706", label="Measured cathode OCP",
)
axes[0].set(xlabel="Full-cell SoC [%]", ylabel="Voltage [V]", title="qOCV reconstruction")
axes[0].legend(frameon=False)

axes[1].plot(
    SOC_GRID * 100, 1000 * (ai2020_ocv - low_rate_data["qocv"]),
    linestyle="none", marker="s", markevery=45, markersize=3.2, color="#6b7280",
)
axes[1].plot(
    SOC_GRID * 100, 1000 * (measured_cathode_ocv - low_rate_data["qocv"]),
    linestyle="none", marker="o", markevery=45, markersize=3.2, color="#d97706",
)
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set(xlabel="Full-cell SoC [%]", ylabel="Model − experiment [mV]", title="qOCV error")
plt.show()

### Chapter 1 결론

최종 조합은 Ai2020 graphite 음극 OCP와 실측 양극 GITT 평균 OCP를 사용한다. C/50 qOCV 기반 window를 유지하고, 이후 동일 동적 데이터에서 geometry, 확산 및 반응속도를 반영한 모델의 전압 정합도를 확인한다.

# Chapter 2. 실험 기반 DFN 파라미터화 및 동일 데이터 정합도 평가

## 2.1 보정 과정 요약

1. PyBaMM Ai2020 parameter set과 전해질 확산계수 단위 확인
2. teardown/SEM 기반 geometry 적용
3. Ai2020 음극 OCP와 실측 양극 GITT OCP 적용
4. C/50 qOCV 기반 stoichiometry window와 inventory 적용
5. GITT 기반 고체 확산계수 적용
6. 최종 선정한 exchange-current-density prefactor coefficient 적용
7. 실제 충·방전 이력으로 0.5C·1C·2C 전압 정합도 확인

## 2.2 Ai2020 셀 teardown 및 전극별 평가

### (1) SEM 기반 전극 기하학

- 음극 평균 입자반경: 3.692 µm
- 양극 평균 입자반경: 4.239 µm
- 음극/양극 두께: 76.5/68.0 µm
- 음극/양극 활물질 체적분율: 0.610/0.620
- 총 겹침면적: 0.0787236 m²

### (2) 양·음극 OCP 측정 및 선정

- 음극: Ai2020 graphite OCP
- 양극: 실측 GITT 반복시험의 평균 OCP

### (3) 저율 충·방전 기반 stoichiometry window

C/50 qOCV로 결정한 `x0`, `x100`, `y0`, `y100`을 사용한다. 최대 고체농도는 독립 측정값이 아니라 선택 window, 기준 용량, 면적, 두께와 활물질 분율을 일관되게 연결한 계산값이다.

### (4) GITT 기반 고체 확산계수 평가

- 음극: 4.170774×10⁻¹⁴ m² s⁻¹
- 양극: 6.888175×10⁻¹⁴ m² s⁻¹

### (5) 양극 반응속도 coefficient

양극 값 `4.37×10⁻⁷`은 최종 A 모델에서 선정한 **Ai2020 exchange-current-density prefactor coefficient convention** 값이다. 기존 Ai2020 함수 형태를 유지하고 prefactor 비율만 조정하며, 이 노트북에서 원 데이터 계보를 다시 산정하지는 않는다. m/s 단위의 직접 측정 반응속도 상수로 해석하지 않는다.

### (6) 음극 반응속도 coefficient

음극 값 `5.54×10⁻⁷`은 **July legacy DRT dataset/convention에서 산정되어 최종 선택한 coefficient**이다. 최신 local-minimum DRT 재산정값과는 구분하며, 최종 A 모델을 동일하게 재현하기 위해 이 값을 유지한다. 이 값도 같은 Ai2020 coefficient convention이며 m/s가 아니다.

## 2.3 제한적 fitting 전략

최종 재현 노트북에서는 최적화나 민감도 탐색을 다시 수행하지 않는다. OCP, window, geometry, 확산계수와 반응속도는 최종 선정값으로 고정하고, 동일한 실험 이력에서 전압과 내부 전위를 다시 계산한다. 이 방식은 전압을 억지로 맞추기 위한 다변수 상호보상을 피하고 재현 절차를 명확하게 유지한다.

## 2.4 최종 모델 구성 코드

아래 셀은 PyBaMM 공식 흐름에 맞춰 `ParameterValues → DFN → Experiment → Simulation → IDAKLUSolver` 순서로 모델을 구성한다.

초기 고체 농도는 `x100`/`y100`의 균일 상태로 시작한다. 원 실험의 선행 partial charge–CV–rest 이력은 생략하고, dynamic workbook의 step 5 저율 방전을 실측 전류로 3.0 V까지 실행한 뒤 10분 rest로 첫 0.5C 충전의 초기 상태를 구성한다. 따라서 이 결과는 전체 실험 선행 이력을 독립적으로 재현한 holdout validation이 아니다.

참고: [PyBaMM 26.8.0 Public API](https://docs.pybamm.org/en/pybamm-v26.8.0.0/source/user_guide/fundamentals/public_api.html) — `ParameterValues`, `Experiment`, `Simulation`과 권장 solver인 `IDAKLUSolver`의 공식 사용 방식.

In [ ]:
NEGATIVE_ELECTRODE_THICKNESS_M = 76.5e-6
POSITIVE_ELECTRODE_THICKNESS_M = 68.0e-6
NEGATIVE_ACTIVE_MATERIAL_FRACTION = 0.610
POSITIVE_ACTIVE_MATERIAL_FRACTION = 0.620
NEGATIVE_PARTICLE_RADIUS_M = 3.692e-6
POSITIVE_PARTICLE_RADIUS_M = 4.239e-6
OVERLAP_AREA_M2 = 45.4e-3 * 51.0e-3 * 34.0

negative_capacity_ah = reference_capacity_ah / (final_window["x100"] - final_window["x0"])
positive_capacity_ah = reference_capacity_ah / (final_window["y0"] - final_window["y100"])
maximum_negative_concentration = 3600.0 * negative_capacity_ah / (
    FARADAY_CONSTANT * OVERLAP_AREA_M2 * NEGATIVE_ELECTRODE_THICKNESS_M * NEGATIVE_ACTIVE_MATERIAL_FRACTION
)
maximum_positive_concentration = 3600.0 * positive_capacity_ah / (
    FARADAY_CONSTANT * OVERLAP_AREA_M2 * POSITIVE_ELECTRODE_THICKNESS_M * POSITIVE_ACTIVE_MATERIAL_FRACTION
)

parameter_values = pybamm.ParameterValues("Ai2020")
original_electrolyte_diffusivity = parameter_values["Electrolyte diffusivity [m2.s-1]"]
original_negative_exchange_current = parameter_values["Negative electrode exchange-current density [A.m-2]"]
original_positive_exchange_current = parameter_values["Positive electrode exchange-current density [A.m-2]"]


def corrected_electrolyte_diffusivity(electrolyte_concentration, temperature):
    # Ai2020 implementation returns the source expression in cm²/s; apply the SI conversion once.
    return 1e-4 * original_electrolyte_diffusivity(electrolyte_concentration, temperature)


def measured_positive_ocp(positive_stoichiometry):
    return pybamm.Interpolant(
        measured_cathode_ocp[0],
        measured_cathode_ocp[1],
        positive_stoichiometry,
        name="Measured cathode GITT OCP",
        interpolator="linear",
    )


AI2020_PREFACTOR_COEFFICIENT = FARADAY_CONSTANT * 1e-11
NEGATIVE_PREFACTOR_COEFFICIENT = 5.54e-7
POSITIVE_PREFACTOR_COEFFICIENT = 4.37e-7


def negative_exchange_current_density(electrolyte_concentration, surface_concentration, maximum_concentration, temperature):
    scale = NEGATIVE_PREFACTOR_COEFFICIENT / AI2020_PREFACTOR_COEFFICIENT
    return scale * original_negative_exchange_current(
        electrolyte_concentration, surface_concentration, maximum_concentration, temperature
    )


def positive_exchange_current_density(electrolyte_concentration, surface_concentration, maximum_concentration, temperature):
    scale = POSITIVE_PREFACTOR_COEFFICIENT / AI2020_PREFACTOR_COEFFICIENT
    return scale * original_positive_exchange_current(
        electrolyte_concentration, surface_concentration, maximum_concentration, temperature
    )


parameter_values.update({
    "Nominal cell capacity [A.h]": reference_capacity_ah,
    "Ambient temperature [K]": 298.15,
    "Initial temperature [K]": 298.15,
    "Initial concentration in electrolyte [mol.m-3]": 1000.0,
    "Negative electrode thickness [m]": NEGATIVE_ELECTRODE_THICKNESS_M,
    "Positive electrode thickness [m]": POSITIVE_ELECTRODE_THICKNESS_M,
    "Negative electrode active material volume fraction": NEGATIVE_ACTIVE_MATERIAL_FRACTION,
    "Positive electrode active material volume fraction": POSITIVE_ACTIVE_MATERIAL_FRACTION,
    "Negative particle radius [m]": NEGATIVE_PARTICLE_RADIUS_M,
    "Positive particle radius [m]": POSITIVE_PARTICLE_RADIUS_M,
    "Electrode height [m]": 51.0e-3,
    "Electrode width [m]": 45.4e-3,
    "Number of electrodes connected in parallel to make a cell": 34.0,
    "Maximum concentration in negative electrode [mol.m-3]": maximum_negative_concentration,
    "Maximum concentration in positive electrode [mol.m-3]": maximum_positive_concentration,
    "Initial concentration in negative electrode [mol.m-3]": final_window["x100"] * maximum_negative_concentration,
    "Initial concentration in positive electrode [mol.m-3]": final_window["y100"] * maximum_positive_concentration,
    "Positive electrode OCP [V]": measured_positive_ocp,
    "Negative particle diffusivity [m2.s-1]": 4.1707741459682473e-14,
    "Positive particle diffusivity [m2.s-1]": 6.888175323807748e-14,
    "Negative electrode exchange-current density [A.m-2]": negative_exchange_current_density,
    "Positive electrode exchange-current density [A.m-2]": positive_exchange_current_density,
    "Negative electrode Bruggeman coefficient (electrolyte)": 2.914,
    "Separator Bruggeman coefficient (electrolyte)": 1.500,
    "Positive electrode Bruggeman coefficient (electrolyte)": 1.830,
    "Electrolyte diffusivity [m2.s-1]": corrected_electrolyte_diffusivity,
})

parameter_summary = pd.DataFrame([
    ["Reference capacity", reference_capacity_ah, "Ah"],
    ["x0 / x100", f"{final_window['x0']:.9f} / {final_window['x100']:.9f}", "-"],
    ["y0 / y100", f"{final_window['y0']:.9f} / {final_window['y100']:.9f}", "-"],
    ["Maximum negative concentration", maximum_negative_concentration, "mol m⁻³"],
    ["Maximum positive concentration", maximum_positive_concentration, "mol m⁻³"],
    ["Negative solid diffusivity", "4.170774e-14", "m² s⁻¹"],
    ["Positive solid diffusivity", "6.888175e-14", "m² s⁻¹"],
    ["Negative exchange-current prefactor coefficient", "5.540000e-7", "Ai2020 convention; July legacy DRT selection"],
    ["Positive exchange-current prefactor coefficient", "4.370000e-7", "Ai2020 convention; final selected value"],
    ["Negative / separator / positive Bruggeman", "2.914 / 1.500 / 1.830", "-"],
], columns=["Parameter", "Value", "Unit or convention"])
display(parameter_summary)

In [ ]:
def evaluate_diffusivity(function, concentration=1000.0, temperature=298.15):
    expression = function(pybamm.Scalar(concentration), pybamm.Scalar(temperature))
    return float(np.asarray(expression.evaluate()).reshape(-1)[0])


raw_diffusivity = evaluate_diffusivity(original_electrolyte_diffusivity)
corrected_diffusivity = evaluate_diffusivity(corrected_electrolyte_diffusivity)
print(f"Ai2020 source expression at 1000 mol m⁻³, 298.15 K: {raw_diffusivity:.6e}")
print(f"After one ×1e-4 SI conversion:                  {corrected_diffusivity:.6e} m² s⁻¹")
print(f"Applied ratio:                                 {corrected_diffusivity / raw_diffusivity:.6e}")

if not np.isclose(corrected_diffusivity / raw_diffusivity, 1e-4, rtol=1e-12):
    raise AssertionError("Electrolyte diffusivity conversion was not applied exactly once.")
if not (1e-12 < corrected_diffusivity < 1e-8):
    raise AssertionError("Corrected electrolyte diffusivity is outside a broad SI sanity range.")

In [ ]:
protocol_steps = [
    f"Discharge at {dynamic_protocol_currents['initial_discharge_a']} A until 3.0 V",
    "Rest for 10 minutes",
]
branch_step_index = {}

for c_rate in TEST_C_RATES:
    charge_current = experimental_branches[(c_rate, True)]["current_a"]
    discharge_current = experimental_branches[(c_rate, False)]["current_a"]
    branch_step_index[(c_rate, True)] = len(protocol_steps)
    protocol_steps.extend([
        f"Charge at {charge_current} A until 4.2 V",
        f"Hold at 4.2 V until {dynamic_protocol_currents['cv_cutoff_a'][c_rate]} A",
        "Rest for 10 minutes",
    ])
    branch_step_index[(c_rate, False)] = len(protocol_steps)
    protocol_steps.extend([
        f"Discharge at {discharge_current} A until 3.0 V",
        "Rest for 10 minutes",
    ])

model = pybamm.lithium_ion.DFN(options={"thermal": "isothermal"})
experiment = pybamm.Experiment([tuple(protocol_steps)], period="10 seconds")
solver = pybamm.IDAKLUSolver(rtol=1e-6, atol=1e-8)

simulation = pybamm.Simulation(
    model,
    parameter_values=parameter_values,
    experiment=experiment,
    solver=solver,
    var_pts={"x_n": 15, "x_s": 15, "x_p": 15, "r_n": 15, "r_p": 15},
)

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=pybamm.SolverWarning)
    solution = simulation.solve()

print("DFN solve completed.")

In [ ]:
def solution_entries(step_solution, variable_name):
    return np.asarray(step_solution[variable_name].entries, dtype=float).squeeze()


cycle = solution.cycles[0]
model_branches = {}
for c_rate in TEST_C_RATES:
    for is_charge in (True, False):
        step_solution = cycle.steps[branch_step_index[(c_rate, is_charge)]]
        time_seconds = solution_entries(step_solution, "Time [s]")
        time_seconds = time_seconds - time_seconds[0]
        negative_stoichiometry = solution_entries(step_solution, "Negative electrode stoichiometry")
        model_branches[(c_rate, is_charge)] = {
            "time_min": time_seconds / 60.0,
            "voltage_v": solution_entries(step_solution, "Terminal voltage [V]"),
            "soc_pct": 100.0 * (
                (negative_stoichiometry - final_window["x0"])
                / (final_window["x100"] - final_window["x0"])
            ),
            "anode_potential_mv": 1000.0 * solution_entries(
                step_solution,
                "Negative electrode surface potential difference at separator interface [V]",
            ),
        }

print("Extracted branches:", sorted(model_branches))

## 2.5 동일 데이터 전압 정합도

모든 표는 같은 네 지표를 사용한다.

- **Common 0–99.5% RMSE/MAE:** 실험과 모델이 함께 존재하는 공통 시간구간의 0–99.5%
- **Time 10–90% RMSE/MAE:** 각 실험 branch elapsed time의 10–90%

통합 지표는 각 조건을 동일한 가중치로 집계한다. RMSE는 조건별 RMSE의 root-mean-square, MAE는 조건별 MAE의 산술평균이다. 사용한 데이터는 파라미터 선정과 겹치므로 독립 holdout validation이 아니다. 모델이 실험 90% 시간 이전에 종료되면 해당 지표를 계산할 수 없으므로 계산을 중단한다.

In [ ]:
def branch_metrics(experiment_branch, model_branch):
    experiment_time = np.asarray(experiment_branch["time_min"], dtype=float)
    experiment_time = experiment_time - experiment_time[0]
    experiment_voltage = np.asarray(experiment_branch["voltage_v"], dtype=float)
    model_time = np.asarray(model_branch["time_min"], dtype=float)
    model_time = model_time - model_time[0]
    model_voltage = np.asarray(model_branch["voltage_v"], dtype=float)

    time_10 = 0.10 * experiment_time[-1]
    time_90 = 0.90 * experiment_time[-1]
    if model_time[-1] < time_90:
        raise RuntimeError(
            f"Model ended at {model_time[-1]:.3f} min before the experiment 90% time {time_90:.3f} min."
        )

    center_grid = np.linspace(time_10, time_90, 401)
    common_end = 0.995 * min(experiment_time[-1], model_time[-1])
    full_grid = np.linspace(0.0, common_end, 401)

    def error_mv(time_grid):
        return 1000.0 * (
            np.interp(time_grid, model_time, model_voltage)
            - np.interp(time_grid, experiment_time, experiment_voltage)
        )

    center_error = error_mv(center_grid)
    full_error = error_mv(full_grid)
    return {
        "Common0_99p5_RMSE_mV": float(np.sqrt(np.mean(full_error**2))),
        "Common0_99p5_MAE_mV": float(np.mean(np.abs(full_error))),
        "Time10_90_RMSE_mV": float(np.sqrt(np.mean(center_error**2))),
        "Time10_90_MAE_mV": float(np.mean(np.abs(center_error))),
    }


metric_rows = []
for c_rate in TEST_C_RATES:
    for is_charge in (True, False):
        metric_rows.append({
            "Nominal_C_rate": c_rate,
            "Direction": "Charge" if is_charge else "Discharge",
            "Applied_current_A": experimental_branches[(c_rate, is_charge)]["current_a"],
            "Effective_C_rate": experimental_branches[(c_rate, is_charge)]["current_a"] / reference_capacity_ah,
            **branch_metrics(
                experimental_branches[(c_rate, is_charge)],
                model_branches[(c_rate, is_charge)],
            ),
        })

branch_metric_table = pd.DataFrame(metric_rows)
display(branch_metric_table.round({
    "Nominal_C_rate": 1,
    "Applied_current_A": 6,
    "Effective_C_rate": 3,
    "Common0_99p5_RMSE_mV": 3,
    "Common0_99p5_MAE_mV": 3,
    "Time10_90_RMSE_mV": 3,
    "Time10_90_MAE_mV": 3,
}))

In [ ]:
def root_mean_square(values):
    values = np.asarray(values, dtype=float)
    return float(np.sqrt(np.mean(values**2)))


summary_rows = []
for group_name, group in [
    ("Charge", branch_metric_table[branch_metric_table["Direction"] == "Charge"]),
    ("Discharge", branch_metric_table[branch_metric_table["Direction"] == "Discharge"]),
    ("All", branch_metric_table),
]:
    summary_rows.append({
        "Group": group_name,
        "Common0_99p5_RMSE_mV": root_mean_square(group["Common0_99p5_RMSE_mV"]),
        "Common0_99p5_MAE_mV": float(group["Common0_99p5_MAE_mV"].mean()),
        "Time10_90_RMSE_mV": root_mean_square(group["Time10_90_RMSE_mV"]),
        "Time10_90_MAE_mV": float(group["Time10_90_MAE_mV"].mean()),
    })

summary_metric_table = pd.DataFrame(summary_rows)
display(summary_metric_table.round(3))

In [ ]:
figure, axes = plt.subplots(2, 3, figsize=(14, 7), sharey=True, constrained_layout=True)
for row, (is_charge, direction) in enumerate([(True, "Charge"), (False, "Discharge")]):
    for column, c_rate in enumerate(TEST_C_RATES):
        axis = axes[row, column]
        experiment_branch = experimental_branches[(c_rate, is_charge)]
        model_branch = model_branches[(c_rate, is_charge)]

        axis.plot(
            experiment_branch["time_min"],
            experiment_branch["voltage_v"],
            color="black",
            linewidth=1.7,
            label="Experiment",
        )
        marker_spacing = max(1, len(model_branch["time_min"]) // 45)
        axis.plot(
            model_branch["time_min"],
            model_branch["voltage_v"],
            linestyle="none",
            marker="o",
            markevery=marker_spacing,
            markersize=3.2,
            color="#d97706",
            label="DFN model",
        )
        axis.axvspan(
            0.10 * experiment_branch["time_min"][-1],
            0.90 * experiment_branch["time_min"][-1],
            color="#9ca3af",
            alpha=0.12,
        )
        axis.set_title(f"{c_rate:g}C {direction}")
        axis.set_xlabel("Elapsed time [min]")
        if column == 0:
            axis.set_ylabel("Terminal voltage [V]")

handles, labels = axes[0, 0].get_legend_handles_labels()
figure.legend(handles, labels, loc="upper center", ncol=2, frameon=False)
figure.suptitle("Experiment and DFN voltage", y=1.03)
plt.show()

## 2.6 SoC별 음극 전위 — 충전 프로토콜 적용 전 사후 확인

아래 전위는 `Negative electrode surface potential difference at separator interface`의 DFN 예측값이다. 기준전극 측정값과 직접 비교된 결과가 아니므로, 충전 제약조건으로 사용할 때는 파라미터 불확실성과 안전 여유를 함께 고려해야 한다.

In [ ]:
figure, axis = plt.subplots(figsize=(8, 5), constrained_layout=True)
anode_rows = []
for c_rate, color in [(0.5, "#2563eb"), (1.0, "#059669"), (2.0, "#dc2626")]:
    branch = model_branches[(c_rate, True)]
    order = np.argsort(branch["soc_pct"], kind="stable")
    soc = np.asarray(branch["soc_pct"])[order]
    anode_potential = np.asarray(branch["anode_potential_mv"])[order]
    unique_soc, unique_index = np.unique(soc, return_index=True)
    unique_potential = anode_potential[unique_index]
    keep = (unique_soc >= 10.0) & (unique_soc <= 59.0)
    axis.plot(unique_soc[keep], unique_potential[keep], color=color, lw=2, label=f"{c_rate:g}C")
    anode_rows.append({
        "C_rate": c_rate,
        "Minimum_anode_potential_10_59_mV": float(np.min(unique_potential[keep])),
    })

axis.axhline(0, color="black", linestyle="--", linewidth=1)
axis.set(
    xlabel="Model full-cell SoC [%]",
    ylabel="Negative electrode potential vs Li/Li+ [mV]",
    title="Predicted negative electrode potential during charge",
)
axis.legend(frameon=False)
plt.show()
display(pd.DataFrame(anode_rows).round(3))

# 최종 선정 모델 및 결론

## 최종 모델

- **기반 모델:** PyBaMM Ai2020 DFN, isothermal
- **OCP:** Ai2020 graphite 음극 + 실측 양극 GITT 평균
- **Window:** C/50 qOCV 기반 최종 stoichiometry window
- **Geometry:** teardown/SEM 기반
- **Diffusion:** GITT 기반 음극·양극 고체 확산계수
- **Kinetics:** Ai2020 exchange-current-density prefactor coefficient convention. 음극은 July legacy DRT dataset/convention에서 산정한 최종 선택값, 양극은 최종 선정값
- **Electrolyte:** Ai2020 함수에 단위 변환 `×1e-4`를 한 번 적용
- **Bruggeman:** 음극/분리막/양극 = 2.914/1.500/1.830

## 주요 성과

이 노트북은 이전 실행 숫자와 자동 비교하지 않고, 현재 입력에서 새로 계산한 조건별 및 통합 지표를 제시한다. 초기 방전과 각 충전 CV 종료전류도 dynamic workbook의 실제 step/record에서 읽어 연속 프로토콜을 구성한다. 이는 동일 데이터 정합도 결과이며 독립 holdout 평가가 아니다.

## 남은 결정

음극 전위는 내부 예측량이며 기준전극 측정값과 직접 비교되지 않았다. 이후 충전 프로토콜 최적화에서는 전압 정합도뿐 아니라 mesh, 초기 상태, 파라미터 민감도와 안전 여유를 함께 검토한다.

In [ ]:
OUTPUT_DIRECTORY = Path.cwd()
branch_metric_table.to_csv(OUTPUT_DIRECTORY / "DFN_voltage_metrics_by_condition.csv", index=False)
summary_metric_table.to_csv(OUTPUT_DIRECTORY / "DFN_voltage_metrics_summary.csv", index=False)
parameter_summary.to_csv(OUTPUT_DIRECTORY / "DFN_parameter_summary.csv", index=False)
print("Saved:")
print("- DFN_voltage_metrics_by_condition.csv")
print("- DFN_voltage_metrics_summary.csv")
print("- DFN_parameter_summary.csv")